# Depth Estimation — Adaptive Road Vision

Estimate monocular depth and approximate object distance for the road-vision pipeline.

### Workflow
1. Load the final test images and the detector selected in Notebook 21.
2. Load a pretrained **Depth Anything V2 Small** monocular depth model through Hugging Face Transformers.
3. Generate dense relative-depth maps.
4. Convert relative depth into an approximate distance representation using an optional calibration scale.
5. Reuse detector bounding boxes and estimate object depth at the center/robust interior of each box.
6. Save per-object distance estimates and visualization samples.

> **Important:** monocular depth without camera calibration provides relative depth rather than guaranteed metric distance. The notebook therefore reports relative depth by default and only reports metric distance when a project calibration scale is explicitly supplied.


In [ ]:
from pathlib import Path
import os, json, math
import numpy as np
import pandas as pd
from PIL import Image
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}. Update PROJECT_ROOT to your actual project folder."
    )
os.chdir(PROJECT_ROOT)


def resolve_dataset_root():
    candidates = [
        PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
        PROJECT_ROOT / "datasets" / "final_condition_balanced",
        PROJECT_ROOT / "datasets" / "final",
    ]
    for candidate in candidates:
        if (candidate / "metadata" / "final_dataset_metadata.csv").is_file():
            return candidate
    for candidate in candidates:
        if any((candidate / "images" / split).is_dir() for split in ("test", "testing")):
            return candidate
    # Last-resort discovery: find the final metadata filename anywhere under datasets.
    datasets_dir = PROJECT_ROOT / "datasets"
    if datasets_dir.is_dir():
        matches = sorted(datasets_dir.rglob("final_dataset_metadata.csv"))
        if matches:
            # metadata/<file> => dataset root is its grandparent
            return matches[0].parent.parent
    return candidates[0]


FINAL_DIR = resolve_dataset_root()
DATASET_ROOT = FINAL_DIR
METADATA_CANDIDATES = [
    DATASET_ROOT / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2" / "metadata" / "final_dataset_metadata.csv",
    PROJECT_ROOT / "datasets" / "processed" / "metadata" / "bdd100k_processed_metadata.csv",
]
METADATA_PATH = next((p for p in METADATA_CANDIDATES if p.is_file()), None)
if METADATA_PATH is None:
    for pattern in ("final_dataset_metadata.csv", "bdd100k_processed_metadata.csv"):
        matches = list((PROJECT_ROOT / "datasets").rglob(pattern)) if (PROJECT_ROOT / "datasets").exists() else []
        if matches:
            METADATA_PATH = matches[0]
            break
if METADATA_PATH is None:
    raise FileNotFoundError(
        "Could not find dataset metadata CSV. Expected final_dataset_metadata.csv "
        "under datasets/<final dataset>/metadata/."
    )

SELECTION_PATH = PROJECT_ROOT / "results" / "detection" / "final_detector_selection.json"
CONFIG_PATH = PROJECT_ROOT / "configs" / "depth.yaml"
RESULT_ROOT = PROJECT_ROOT / "results" / "depth"
MODEL_ROOT = PROJECT_ROOT / "models" / "depth"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Metadata:", METADATA_PATH)
print("Device:", DEVICE)

## 1. Configuration

The default model is intentionally lightweight enough for a notebook workflow. Increase image size or switch to a larger checkpoint only after confirming the pipeline works.


In [ ]:
MODEL_NAME = "depth-anything/Depth-Anything-V2-Small-hf"

DEPTH_IMAGE_SIZE = 518
MAX_IMAGES = None       # Set an integer for a quick smoke test.
SAVE_DEPTH_MAPS = True
SAVE_VISUALIZATIONS = True
VISUALIZATION_COUNT = 12

# Metric-distance calibration:
# Leave None unless you have a validated project-specific scale.
# If supplied, distance_m = relative_depth * DEPTH_SCALE_M_PER_UNIT.
DEPTH_SCALE_M_PER_UNIT = None

CONF_THRESHOLD = 0.25
DETECTOR_IMAGE_SIZE = 640

print("Depth model:", MODEL_NAME)
print("Metric scale:", DEPTH_SCALE_M_PER_UNIT)


In [ ]:
# Use the detector selected in Notebook 21. Do not silently switch models:
# a fallback would make the downstream experiment inconsistent.
if not SELECTION_PATH.is_file():
    raise FileNotFoundError(
        f"Missing detector selection report: {SELECTION_PATH}. "
        "Run Notebook 21 after training/evaluating YOLO11n and YOLO11s."
    )

selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
selected_detector = selection.get("selected_detector", "Unknown")
selected_model = selection.get("selected_model", "Unknown")
checkpoint_value = (
    selection.get("stable_downstream_checkpoint")
    or selection.get("selected_checkpoint")
    or selection.get("checkpoint_path")
    or selection.get("final_checkpoint")
)
if isinstance(checkpoint_value, dict):
    checkpoint_value = checkpoint_value.get("path") or checkpoint_value.get("checkpoint")
if not checkpoint_value:
    raise KeyError(
        "The detector selection report has no 'stable_downstream_checkpoint' "
        "or 'selected_checkpoint' field. Re-run Notebook 21."
    )
selected_checkpoint = Path(checkpoint_value)
if not selected_checkpoint.is_absolute():
    selected_checkpoint = (PROJECT_ROOT / selected_checkpoint).resolve()
if not selected_checkpoint.is_file():
    raise FileNotFoundError(
        f"Selected detector checkpoint not found: {selected_checkpoint}. "
        "Check Notebook 21's checkpoint path; no fallback model will be used."
    )

print("Selected detector:", selected_detector)
print("Selected model:", selected_model)
print("Checkpoint:", selected_checkpoint)

metadata = pd.read_csv(METADATA_PATH)
if "split" not in metadata.columns:
    raise KeyError(f"Metadata CSV has no 'split' column: {METADATA_PATH}")
if "image_name" not in metadata.columns:
    for alternative in ("filename", "file_name", "image"):
        if alternative in metadata.columns:
            metadata["image_name"] = metadata[alternative].astype(str)
            break
if "image_name" not in metadata.columns and "image_path" not in metadata.columns:
    raise KeyError("Metadata must contain image_name (or filename) or image_path.")
if "project_condition" not in metadata.columns:
    if "condition" in metadata.columns:
        metadata["project_condition"] = metadata["condition"]
    elif "weather" in metadata.columns:
        metadata["project_condition"] = metadata["weather"]
    else:
        metadata["project_condition"] = "Unknown"

split_values = metadata["split"].astype(str).str.strip().str.lower()
test_meta = metadata[split_values.isin(["test", "testing", "test_set"])].copy()
if test_meta.empty:
    raise ValueError(f"No test rows found in metadata column 'split': {METADATA_PATH}")
print("Test rows in metadata:", len(test_meta))

## 2. Install/import depth dependencies

If Transformers is not installed in the project environment, install it once before running the model-loading cell.


In [ ]:
try:
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation
except ImportError as e:
    raise ImportError(
        "Install depth dependencies in the active environment with: "
        "pip install transformers accelerate safetensors"
    ) from e

try:
    import torchvision  # noqa: F401
except Exception:
    # torchvision transforms are not required by this notebook; avoid making
    # inference depend on an optional, sometimes mismatched installation.
    pass

## 3. Load Depth Anything V2

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
depth_model = AutoModelForDepthEstimation.from_pretrained(MODEL_NAME)
depth_model = depth_model.to(DEVICE)
depth_model.eval()
print("Loaded depth model:", MODEL_NAME)

## 4. Load the selected detector

In [ ]:
try:
    from ultralytics import YOLO
except ImportError as e:
    raise ImportError("Install Ultralytics with: pip install ultralytics") from e

detector = YOLO(str(selected_checkpoint))
print("Loaded selected detector:", selected_checkpoint)

## 5. Resolve test-image paths

In [ ]:
def resolve_image(row):
    candidates = []
    for key in ("image_path", "image_name"):
        if key not in row.index or pd.isna(row[key]):
            continue
        value = str(row[key]).strip()
        if not value or value.lower() == "nan":
            continue
        # Accept Windows-style metadata paths even if the notebook is run on another OS.
        value = value.replace("\\", "/")
        p = Path(value)
        if p.is_absolute():
            candidates.append(p)
        else:
            candidates.extend([
                PROJECT_ROOT / p,
                DATASET_ROOT / p,
                DATASET_ROOT / "images" / "test" / p.name,
                DATASET_ROOT / "images" / "testing" / p.name,
            ])
        # If metadata stores a path with a stale root, try its filename under test.
        candidates.extend([
            DATASET_ROOT / "images" / "test" / p.name,
            DATASET_ROOT / "images" / "testing" / p.name,
            DATASET_ROOT / "test" / p.name,
            DATASET_ROOT / "testing" / p.name,
        ])

    name_value = row.get("image_name", row.get("filename", row.get("file_name", "")))
    name = str(name_value) if pd.notna(name_value) else ""
    if name:
        stem = Path(name).stem
        for folder in ("test", "testing"):
            for ext in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
                candidates.extend([
                    DATASET_ROOT / "images" / folder / (stem + ext),
                    DATASET_ROOT / folder / (stem + ext),
                ])
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen and candidate.is_file():
            return candidate.resolve()
        seen.add(key)
    return None

test_meta["resolved_image_path"] = test_meta.apply(resolve_image, axis=1)
missing_count = int(test_meta["resolved_image_path"].isna().sum())
if missing_count:
    print(f"Warning: skipping {missing_count} test rows whose image files could not be resolved.")
test_meta = test_meta[test_meta["resolved_image_path"].notna()].copy()
if MAX_IMAGES is not None:
    test_meta = test_meta.head(int(MAX_IMAGES)).copy()
if test_meta.empty:
    raise FileNotFoundError(
        "No test images could be resolved. Check metadata image_path/image_name "
        "and the dataset images/test folder."
    )
print("Images to process:", len(test_meta))

## 6. Depth inference helpers

Depth Anything V2 outputs a dense **relative depth** representation. The helper normalizes each prediction for visualization while retaining the raw prediction for object-level measurements.


In [ ]:
def infer_depth(image):
    inputs = processor(images=image, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        outputs = depth_model(**inputs)
    predicted = outputs.predicted_depth
    if predicted.ndim == 3:
        predicted = predicted.unsqueeze(1)
    depth = torch.nn.functional.interpolate(
        predicted,
        size=(image.height, image.width),
        mode="bicubic",
        align_corners=False,
    ).squeeze().detach().float().cpu().numpy()
    depth = np.nan_to_num(depth, nan=0.0, posinf=0.0, neginf=0.0)
    return depth.astype(np.float32)


def normalize_depth(depth):
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        return np.zeros_like(depth, dtype=np.float32)
    lo, hi = np.percentile(finite, [2, 98])
    if hi <= lo:
        return np.zeros_like(depth, dtype=np.float32)
    return np.clip((depth - lo) / (hi - lo), 0, 1).astype(np.float32)


def object_depth(depth, xyxy, margin_ratio=0.20):
    h_img, w_img = depth.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    x1, x2 = sorted((max(0.0, min(float(w_img), x1)), max(0.0, min(float(w_img), x2))))
    y1, y2 = sorted((max(0.0, min(float(h_img), y1)), max(0.0, min(float(h_img), y2))))
    w, h = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    xa, ya = int(x1 + margin_ratio * w), int(y1 + margin_ratio * h)
    xb, yb = int(x2 - margin_ratio * w), int(y2 - margin_ratio * h)
    if xb <= xa or yb <= ya:
        xa, ya, xb, yb = int(x1), int(y1), max(int(x2), int(x1) + 1), max(int(y2), int(y1) + 1)
    crop = depth[max(0, ya):min(h_img, yb), max(0, xa):min(w_img, xb)]
    crop = crop[np.isfinite(crop)]
    if crop.size == 0:
        return float("nan")
    return float(np.median(crop))


def relative_to_metric(relative_depth, scale=None):
    # A scale is only valid if calibrated specifically for this model/output.
    if scale is None or not np.isfinite(relative_depth):
        return float("nan")
    return float(relative_depth * scale)

## 7. Run depth estimation and object-distance extraction

For each test image:
- generate a dense depth map,
- run the selected detector,
- estimate depth inside every detected object,
- save object-level records.

The raw relative-depth value is preserved. A metric distance column is only populated if `DEPTH_SCALE_M_PER_UNIT` is set.


In [ ]:
rows = []
depth_map_dir = RESULT_ROOT / "depth_maps"
visual_dir = RESULT_ROOT / "visualizations"
if SAVE_DEPTH_MAPS:
    depth_map_dir.mkdir(parents=True, exist_ok=True)
if SAVE_VISUALIZATIONS:
    visual_dir.mkdir(parents=True, exist_ok=True)

visualization_counter = 0
for idx, row in enumerate(test_meta.itertuples(index=False), start=1):
    image_path = Path(row.resolved_image_path)
    try:
        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")
    except Exception as exc:
        print(f"Skipping unreadable image {image_path}: {exc}")
        continue

    try:
        depth = infer_depth(image)
        depth_norm = normalize_depth(depth)
        prediction = detector.predict(
            source=np.asarray(image),
            imgsz=DETECTOR_IMAGE_SIZE,
            conf=CONF_THRESHOLD,
            device=0 if DEVICE == "cuda" else "cpu",
            verbose=False,
        )[0]
    except Exception as exc:
        print(f"Skipping inference failure for {image_path.name}: {exc}")
        continue

    if prediction.boxes is not None and len(prediction.boxes):
        boxes = prediction.boxes.xyxy.detach().cpu().numpy()
        classes = prediction.boxes.cls.detach().cpu().numpy().astype(int)
        confs = prediction.boxes.conf.detach().cpu().numpy()
    else:
        boxes, classes, confs = np.empty((0, 4)), np.empty((0,), dtype=int), np.empty((0,))

    condition = getattr(row, "project_condition", "Unknown")
    image_name = getattr(row, "image_name", image_path.name)
    image_id = getattr(row, "image_id", image_path.stem)
    for det_idx, (box, cls, conf) in enumerate(zip(boxes, classes, confs)):
        rel_depth = object_depth(depth, box)
        rows.append({
            "image_id": image_id,
            "image_name": image_name,
            "condition": condition if pd.notna(condition) else "Unknown",
            "image_path": str(image_path),
            "detection_index": int(det_idx),
            "class_id": int(cls),
            "class_name": (
                detector.names.get(int(cls), str(cls))
                if isinstance(detector.names, dict)
                else (detector.names[int(cls)] if isinstance(detector.names, (list, tuple)) and int(cls) < len(detector.names) else str(cls))
            ),
            "confidence": float(conf),
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
            "relative_depth": float(rel_depth),
            "distance_m": relative_to_metric(rel_depth, DEPTH_SCALE_M_PER_UNIT),
        })

    if SAVE_DEPTH_MAPS:
        np.save(depth_map_dir / f"{image_path.stem}_depth.npy", depth)
    if SAVE_VISUALIZATIONS and visualization_counter < VISUALIZATION_COUNT:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(image); axes[0].set_title("Original"); axes[0].axis("off")
        axes[1].imshow(depth_norm, cmap="magma"); axes[1].set_title("Normalized relative depth"); axes[1].axis("off")
        fig.suptitle(image_path.name)
        fig.tight_layout()
        fig.savefig(visual_dir / f"{image_path.stem}_depth.png", dpi=140, bbox_inches="tight")
        plt.close(fig)
        visualization_counter += 1
    if idx % 100 == 0 or idx == len(test_meta):
        print(f"Processed {idx:,}/{len(test_meta):,} images; object rows so far: {len(rows):,}")

depth_df = pd.DataFrame(rows, columns=[
    "image_id", "image_name", "condition", "image_path", "detection_index",
    "class_id", "class_name", "confidence", "x1", "y1", "x2", "y2",
    "relative_depth", "distance_m"
])
print("Object-depth rows:", len(depth_df))
display(depth_df.head())

## 8. Depth statistics

In [ ]:
if depth_df.empty:
    print("No detections were produced for the processed test images. Depth maps may still have been saved.")
    print("Check the selected checkpoint, detector class mapping, image quality, and CONF_THRESHOLD.")
    depth_summary = pd.DataFrame(columns=[
        "condition", "images", "detections", "mean_relative_depth",
        "median_relative_depth", "min_relative_depth", "max_relative_depth"
    ])
else:
    depth_summary = (
        depth_df.groupby("condition", dropna=False)
        .agg(
            images=("image_name", "nunique"),
            detections=("detection_index", "count"),
            mean_relative_depth=("relative_depth", "mean"),
            median_relative_depth=("relative_depth", "median"),
            min_relative_depth=("relative_depth", "min"),
            max_relative_depth=("relative_depth", "max"),
        )
        .reset_index()
    )
    display(depth_summary)

## 9. Closest detected objects

Relative depth is model-dependent. The ordering is useful for prioritizing potentially near objects **within the same scene**, but it should not be interpreted as meters without calibration.


In [ ]:
if not depth_df.empty:
    # Depth Anything V2 outputs model-dependent relative depth, not meters.
    # For this checkpoint, larger predicted values are conventionally treated as nearer.
    # Verify visually against saved maps; never interpret these values as calibrated distance.
    closest = (
        depth_df.sort_values(["image_name", "relative_depth"], ascending=[True, False])
        .groupby("image_name", as_index=False, dropna=False)
        .head(3)
    )
    display(closest[[
        "image_name", "condition", "class_name", "confidence", "relative_depth", "distance_m"
    ]].head(30))
else:
    print("Closest-object table unavailable because there are no detections.")

## 10. Optional calibration check

If you have a validated camera/depth calibration for this project, set `DEPTH_SCALE_M_PER_UNIT` near the top and rerun the inference cell.

Without that calibration, the notebook deliberately leaves `distance_m` as NaN.


In [ ]:
print("Metric scale:", DEPTH_SCALE_M_PER_UNIT)

if DEPTH_SCALE_M_PER_UNIT is None:
    print(
        "Metric distance is disabled. Current output is relative depth only."
    )
else:
    print(
        "Metric distance enabled using scale:",
        DEPTH_SCALE_M_PER_UNIT,
        "meters per relative-depth unit"
    )


## 11. Save depth-estimation artifacts

In [ ]:
object_csv = RESULT_ROOT / "depth_object_estimates.csv"
summary_csv = RESULT_ROOT / "depth_condition_summary.csv"
report_json = RESULT_ROOT / "depth_estimation_report.json"
depth_df.to_csv(object_csv, index=False)
depth_summary.to_csv(summary_csv, index=False)

report = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "metadata_path": str(METADATA_PATH),
    "selected_detector": selected_detector,
    "selected_model": selected_model,
    "selected_checkpoint": str(selected_checkpoint),
    "depth_model": MODEL_NAME,
    "device": DEVICE,
    "test_metadata_rows": int(len(test_meta)),
    "test_images_processed": int(test_meta["resolved_image_path"].nunique()),
    "object_depth_rows": int(len(depth_df)),
    "images_with_detections": int(depth_df["image_name"].nunique()) if not depth_df.empty else 0,
    "images_without_detections_or_unmatched_rows": int(max(0, test_meta["resolved_image_path"].nunique() - (depth_df["image_path"].nunique() if not depth_df.empty else 0))),
    "metric_scale_m_per_unit": DEPTH_SCALE_M_PER_UNIT,
    "metric_distance_enabled": DEPTH_SCALE_M_PER_UNIT is not None,
    "depth_map_directory": str(depth_map_dir) if SAVE_DEPTH_MAPS else None,
    "visualization_directory": str(visual_dir) if SAVE_VISUALIZATIONS else None,
    "object_csv": str(object_csv),
    "summary_csv": str(summary_csv),
    "note": (
        "Depth Anything V2 gives model-dependent relative depth, not physical meters. "
        "The distance_m column remains NaN unless a valid project-specific calibration scale is supplied. "
        "Object-depth values are estimates sampled within detector boxes and are not ground-truth distance measurements."
    ),
}
report_json.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved object estimates:", object_csv)
print("Saved condition summary:", summary_csv)
print("Saved report:", report_json)
print("Processed images:", report["test_images_processed"], "| object rows:", report["object_depth_rows"])

## Final output

Main object-level estimates:

```text
results/depth/depth_object_estimates.csv
```

Condition summary and run report:

```text
results/depth/depth_condition_summary.csv
results/depth/depth_estimation_report.json
```

Dense depth maps are saved under `results/depth/depth_maps/`, and sample visualizations under `results/depth/visualizations/`.

**Interpretation warning:** these are monocular relative-depth estimates, not calibrated distances in meters. Do not feed `distance_m` into the risk engine as a true distance unless a valid calibration has been measured and validated.